![](https://github.com/ibmm-unibe-ch/FrankenMSA/blob/dev/app/assets/frankenmsa_header.png?raw=true)

# frankenMSA-Local
This notebook launches the [frankenMSA App](https://github.com/ibmm-unibe-ch/FrankenMSA/tree/main/) **in a Local Environment** to provide a GUI for manipulating Multiple Sequence Alignments (MSAs).


Tip: use “Runtime” → “Run all” (or `Ctrl + F9`) to execute all cells.

In [ ]:
#@title Install Prerequisites

INSTALL_PREREQUISITES = False  #@param {type:"boolean"}

import os, subprocess

if INSTALL_PREREQUISITES:
    # Plain Dash only; removed jupyter_dash
    subprocess.run(["pip","install","termcolor","gitpython","ipywidgets","ipython","python-dotenv"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

import git, sys, importlib
from pathlib import Path
from termcolor import colored
import dotenv

dotenv.load_dotenv()
os.environ["FRANKEN_RUNTIME"] = "local"

def warn(msg):
    print(colored("[WARNING] ", "yellow") + msg, file=sys.stderr)

def info(msg):
    print(colored("[INFO] ", "cyan") + msg)

if INSTALL_PREREQUISITES:
    info("Installed prerequisite packages.")

In [ ]:
#@title Prepare Rendering and Sharing options

import ipywidgets as widgets
from IPython.display import display

_render_dropdown = widgets.Dropdown(
    options=[
        ("External browser tab", "external"),
        ("Inline (within notebook)", "inline"),
    ],
    value="external",
    description="Render:",
)
_ngrok_checkbox = widgets.Checkbox(
    value=False,
    description="Create ngrok share link",
)
_port_input = widgets.IntText(
    value=8050,
    description="Port:",
    min=1024,
    max=65535,
)
_status = widgets.Output()

use_ngrok = lambda : _ngrok_checkbox.value

def _enforce_render_on_ngrok(change):
    if change["name"] != "value":
        return
    with _status:
        _status.clear_output()
        if change["new"]:
            if _render_dropdown.value != "external":
                _render_dropdown.value = "external"
            print("ngrok sharing forces external rendering.")
        else:
            print("ngrok sharing disabled; inline available.")

def _enforce_ngrok_on_render(change):
    if change["name"] != "value":
        return
    if use_ngrok() and change["new"] != "external":
        with _status:
            _status.clear_output()
            print("ngrok sharing forces external rendering.")
        _render_dropdown.value = "external"

_ngrok_checkbox.observe(_enforce_render_on_ngrok, names="value")
_render_dropdown.observe(_enforce_ngrok_on_render, names="value")
display(widgets.VBox([_render_dropdown, _ngrok_checkbox, _port_input, _status]))

In [ ]:
#@title Install frankenMSA

INSTALL_FRANKENMSA = False #@param {type:"boolean"}

if INSTALL_FRANKENMSA:
    setup_py = Path("setup.py")
    import subprocess
    if setup_py.exists():
        subprocess.run(["pip","install","-e","."], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        info("frankenMSA installed.")
    else:
        warn("frankenMSA setup.py not found; this does not look like the frankenMSA repository. Cloning from GitHub...")
        FRANKEN_GIT_URL = "https://github.com/ibmm-unibe-ch/FrankenMSA.git"
        FRANKEN_GIT_BRANCH = input("Enter the frankenMSA branch to clone (leave empty for default: main): ").strip() or "main"
        info(f"Cloning branch '{FRANKEN_GIT_BRANCH}'")
        subprocess.run(["git","clone","--branch", FRANKEN_GIT_BRANCH, FRANKEN_GIT_URL, "FrankenMSA"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        subprocess.run("cd FrankenMSA; pip install -e . > /dev/null 2>&1; cd -; mv FrankenMSA/app .", shell=True, check=True)
    info("frankenMSA installation complete.")
    
# kill any existing instances
!pkill -f "app/app.py" 2>/dev/null || true
!pkill -f "gunicorn" 2>/dev/null || true
!pkill -f "ngrok" 2>/dev/null || true

In [ ]:
#@title Launch FrankenMSA App (without ngrok sharing)
from frankenmsa.runtime import normalize_runtime_environment

if not use_ngrok():
    normalize_runtime_environment(runtime="local")
    from app.app import launch
    use_inline = _render_dropdown.value == "inline"
    host = "0.0.0.0" if use_inline else "localhost"
    launch(
        render_mode=_render_dropdown.value,
        port=_port_input.value,
        host=host,
        runtime="local",
    )

In [ ]:
#@title Launch frankenMSA App (with ngrok sharing)
if use_ngrok():

    try:
        import pyngrok
    except:
        info("Installing pyngrok...")
        subprocess.run(["pip","install","pyngrok"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        info("pyngrok installed.")
    try:
        from pyngrok import ngrok

    except:
        raise ImportError("Pyngrok could not be installed")

    from pyngrok import ngrok, conf
    from frankenmsa.runtime import build_app_launch_env, normalize_runtime_environment
    import getpass, re
    import dotenv
    dotenv.load_dotenv()
    normalize_runtime_environment(runtime="local")
    
    token = os.environ.get("NGROK_AUTH_TOKEN", "").strip()
    if not token:
        warn("No ngrok auth token found in NGROK_AUTH_TOKEN env variable.")
        warn("You can sign up for a free ngrok account at https://ngrok.com/")
        warn("To avoid entering the token every time, the token is set it in the NGROK_AUTH_TOKEN environment variable after entering.")

        token = getpass.getpass("Enter ngrok authtoken (hidden): ").strip().strip("'").strip('"')
        os.environ["NGROK_AUTH_TOKEN"] = token

        info("ngrok auth token set as environment variable.")
    conf.get_default().auth_token = token

    public_url = None
    PORT = _port_input.value
    try:
        for t in ngrok.get_tunnels():
            addr = (t.config or {}).get("addr", "")
            if addr.endswith(f":{PORT}"):
                public_url = t.public_url
                print("♻️ Reusing existing tunnel:", public_url)
                break

        if not public_url:
            tun = ngrok.connect(addr=f"0.0.0.0:{PORT}", proto="http")
            public_url = tun.public_url
            print("✅ Created new tunnel:", public_url)

    except Exception as e:
        msg = str(e)
        m = re.search(r"https?://[a-z0-9\-]+\.ngrok-[\w\-]+\.(?:dev|app)", msg)
        if m:
            public_url = m.group(0)
            warn("♻️ Using tunnel from error message:", public_url)
        else:
            raise e

    import subprocess, time

    env = build_app_launch_env(
        port=PORT,
        host=os.environ.get("HOST", "0.0.0.0"),
        render_mode=_render_dropdown.value,
        runtime="local",
        use_ngrok=True,
        public_url=public_url,
        base_env=os.environ,
    )

    proc = subprocess.Popen(
        [sys.executable, "app/app.py"],
        cwd=str(Path.cwd().resolve()),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    start = time.time()
    lines = []
    while time.time() - start < 25:
        ln = proc.stdout.readline()
        if ln:
            lines.append(ln.rstrip())
            if "Running on" in ln or "Dash is running" in ln:
                break
        else:
            time.sleep(0.2)

    display_host = env["HOST"] if env["HOST"] not in {"0.0.0.0", "::"} else "0.0.0.0"
    open_target = public_url if public_url else f"http://{display_host}:{PORT}"

    print("\n---- recent logs ----")
    print("\n".join(lines[-20:]))
    print("---------------------")
    info(f"🌐 Open: {open_target}")
    print("📡 Tailing frankenMSA app logs (Ctrl+C to stop):")
    while True:
        line = proc.stdout.readline()
        if not line:
            time.sleep(0.2)
            continue
        print(line, end="")